In [1]:
from qiskit_metal import designs, Dict
from qiskit_metal.qlibrary.tlines.meandered_grounded import RouteMeanderGrounded
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround

TOTAL_LENGTH_MM = 14.9669

In [2]:
design = designs.DesignPlanar({}, overwrite_enabled=True)
total_length_mm=TOTAL_LENGTH_MM
design.chips.main.size.size_x = '4.8mm'
design.chips.main.size.size_y = '2.4mm'
design.chips.main.size.size_z = '500um'
design.chips.main.size.center_x = '0mm'
design.chips.main.size.center_y = '-1mm'

design.variables['cpw_width'] = '10 um'
design.variables['cpw_gap'] = '6 um'

x1, y1 = '-2mm', '0mm'
launch_options1 = dict(chip='main', pos_x=x1, pos_y=y1, orientation='360',
                        lead_length='30um', pad_height='103um',
                        pad_width='103um', pad_gap='60um')
LP1 = LaunchpadWirebond(design, 'LP1', options=launch_options1)

x2 = '2mm'
launch_options2 = dict(chip='main', pos_x=x2, pos_y=y1, orientation='180',
                        lead_length='30um', pad_height='103um',
                        pad_width='103um', pad_gap='60um')
LP2 = LaunchpadWirebond(design, 'LP2', options=launch_options2)

TL = RoutePathfinder(design, 'TL', options=dict(
    chip='main', trace_width='10um', trace_gap='6um', fillet='90um',
    hfss_wire_bonds=True, lead=dict(end_straight='0.1mm'),
    pin_inputs=Dict(
        start_pin=Dict(component='LP1', pin='tie'),
        end_pin=Dict(component='LP2', pin='tie'))))

# Both ends OPEN (this is what makes it a half-wave resonator, and
# what the S21 driven sweep request means by "both ends open")
otg1 = OpenToGround(design, 'otg1', options=dict(
    chip='main', pos_x='-0.2mm', pos_y='-40um', orientation=180))
otg2 = OpenToGround(design, 'otg2', options=dict(
    chip='main', pos_x='0mm', pos_y='-1.35mm', orientation=-90))

common_kwargs = dict(
    trace_width='10um',
    trace_gap='6um',
    total_length=f'{total_length_mm}mm',
    hfss_wire_bonds=False,
    fillet='99.9 um',
    lead=dict(start_straight='300um'),
    pin_inputs=Dict(
        start_pin=Dict(component='otg1', pin='open'),
        end_pin=Dict(component='otg2', pin='open')),
)

res1 = RouteMeander(design, 'resonator1', Dict(**common_kwargs))

In [4]:
from qiskit_metal import MetalGUI

gui = MetalGUI(design)

In [5]:
# Funciones para el cálculo de parámetros en el circuito
from design_helper import DesignHelper
from chip import Chip
from components import Resonator, Qubit

# inicializamos el helper
k_inductance_ratio = 0.00
helper = DesignHelper(alpha_inductance = k_inductance_ratio)
film_thickness = helper.ureg.Quantity(200, "nm")

In [6]:
import numpy as np
from scipy.optimize import minimize_scalar, brentq
import pandas as pd

def estimate_cpw_gap(dh, 
                                target_Z0=50.0,    
                                trace_width=10.0,    
                                thickness=60e-3,     # Kept your 60nm default
                                eps_r=11.9): # Added to support lambda/4, lambda/2, etc.
    """
    Numerically inverts the DesignHelper analytical models to find the 
    geometric parameters for a CPW resonator of arbitrary wavelength ratio.
    """
    print(f"--- Target: Z0={target_Z0} Ω")
    
    # ==========================================
    # 1. Solve for CPW Gap (g) to hit Target Z0
    # ==========================================
    def impedance_error(g_guess):
        params = dh.compute_cpw_geometric_parameters(
            width=trace_width, 
            gap=g_guess, 
            thickness=thickness,
            eps_r=eps_r
        )
        Z0_guess = dh._as_quantity(params['Z0'], 'ohm').magnitude
        return abs(Z0_guess - target_Z0)
    
    res_gap = minimize_scalar(impedance_error, bounds=(1.0, 20.0), method='bounded')
    optimal_g = res_gap.x
    
    # Extract the final phase velocity 
    final_params = dh.compute_cpw_geometric_parameters(
        width=trace_width, 
        gap=optimal_g, 
        thickness=thickness,
        eps_r=eps_r
    )
    
    Z0_final = dh._as_quantity(final_params['Z0'], 'ohm').magnitude
    print(f"1. CPW Geometry: Width = {trace_width} um, Optimal Gap = {optimal_g:.3f} um")
    print(f"   Achieved Z0 = {Z0_final:.2f} Ω")
    
    return optimal_g

def max_distance_ordering(nums):
    """Dada una secuencia de números, crea la secuencia que maximiza las distancia entre elementos semejantes

    Args:
        nums (array): array de números iniciales

    Returns:
        array: array con distancias maximizadas
    """
    nums = sorted(nums)
    result = []
    
    def recurse(lo, hi):
        if lo > hi:
            return
        mid = (lo + hi) // 2
        result.append(nums[mid])
        recurse(lo, mid - 1)
        recurse(mid + 1, hi)
    
    recurse(0, len(nums) - 1)
    return result

def interleave_frequency_groups(freq_lists):
    """
    Given a list of frequency groups (each already sorted),
    interleaves them so that no two frequencies from the same
    group are adjacent. Maximizes inter-group distance.
    
    Example: [[f1, f2], [f3, f4]] -> [f1, f3, f2, f4]
    """
    # Sort each group internally
    groups = [sorted(g) for g in freq_lists]
    result = []
    # Round-robin pick one from each group at a time
    max_len = max(len(g) for g in groups)
    for i in range(max_len):
        for g in groups:
            if i < len(g):
                result.append(g[i])
    return result


In [8]:
# OJO: trabajamos en unidades de frecuencia, NO DE FRECUENCIA ANGULAR

resonator_width = helper.ureg.Quantity(10, "um")

gap = estimate_cpw_gap(helper, target_Z0=50, trace_width= resonator_width, thickness= film_thickness)
resonator_gap = helper.ureg.Quantity(gap, "um")

chi_target_freq = helper.ureg.Quantity(2.0, "MHz")
chi_target_energy = helper._frequency_as_energy(chi_target_freq)

substrate_epsilon = 11.9

cpw_geometric_parameters = helper.compute_cpw_geometric_parameters(resonator_width, resonator_gap, thickness= film_thickness, eps_r= substrate_epsilon)
Cg_cpw = cpw_geometric_parameters["Cg"]
Lg_cpw = cpw_geometric_parameters["Lg"]
Z0_cpw = cpw_geometric_parameters["Z0"]
v_cpw = cpw_geometric_parameters["v"]
eps_eff_cpw = cpw_geometric_parameters["eps_eff"]


--- Target: Z0=50 Ω
1. CPW Geometry: Width = 10 micrometer um, Optimal Gap = 6.430 um
   Achieved Z0 = 50.00 Ω


In [20]:
substrate_epsilon = 11.9
help_dic = helper.compute_cpw_geometric_parameters('10um', '6um', '200nm', substrate_epsilon)

In [21]:
help_dic.keys()

dict_keys(['Cg', 'Lg', 'Z0', 'v', 'eps_eff'])

In [22]:
velocity = help_dic['v']
target_freq = helper.ureg.Quantity(5, 'GHz')

In [23]:
(0.5*velocity / target_freq).to_reduced_units().to('mm')

<Quantity(11.9906802, 'millimeter')>